# Overtaking experiment: 2,250 m / 120 s, 1,000 episodes

Train fresh adaptive and always-on Expected SARSA agents using the same route,
deadline, rewards, features, learning settings, training seed, and evaluation seeds
as `sensor_time_completion_120s_1000episodes.ipynb`.

The traffic distribution changes: replace the 12 random vehicles with three
constant-speed leaders at 12 m/s, initially 60, 180, and 300 m ahead in successive
lanes relative to ego's starting lane. They hold their lanes. These are slower
vehicles from reset, rather than sudden braking events. Staggering leaves room
to change lanes; every lane contains a slower vehicle. Even the farthest leader
travels only 1,740 m from ego's starting position by the deadline, so safe
completion requires moving ahead of all three. The required route average is
18.75 m/s. No passing bonus or privileged traffic input is added.

This controlled scenario tests overtaking under predictable traffic; it does not
establish performance in dense, reactive traffic. The old task remains available
as a control. First evaluate the saved old policies on the new traffic to measure
transfer; then train fresh policies and compare. Run top to bottom with `.venv`.


## 1. Setup

Policies use sensor observations, ego telemetry, and normalized remaining route
and deadline (assumed available navigation/clock telemetry). They receive no
traffic truth or weather labels. Route progress is measured along the road's
world x axis relative to the starting position, not cumulative path length.


In [ ]:
from pathlib import Path
import sys, os, copy, json
from dataclasses import asdict
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "environments/sensor_highway/env.py").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook within MORL4AVSO.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from importlib.metadata import version
from environments.sensor_highway import ExperimentConfig
from algorithms.sensor_agent import AgentSettings, ObservationEncoder, LinearExpectedSarsa, FEATURE_SCHEMA
from experiments.agent_helpers import train_agent, evaluate_agents, run_agent
from experiments.notebook_helpers import plot_snapshot
print("Interpreter:", sys.executable)
PACKAGE_VERSIONS = {name: version(name) for name in ["highway-env", "gymnasium", "numpy", "pandas", "matplotlib"]}
print(PACKAGE_VERSIONS)

## 2. What the agent learns

The original Q-table stored one estimate for each discrete state/action pair.
Here there are too many possible grid states, so we use a **linear value model**:

`Q(observation, action) = action_weights · observation_features`

Each action has a vector of weights. Features summarize obstacle evidence, free
space evidence, measurement support, age, and radial velocity in 15 nearby regions,
plus ego telemetry and sensor status. Unknown cells have no free-space evidence.
This shares experience across similar observations, but it loses scene detail.
No exact lane ID, true gap, or hidden velocity is added.

We use **Expected SARSA**, an on-policy temporal-difference method:

`target = reward + gamma * expected_next_Q_under_epsilon_greedy_policy`

The expectation is `(1-epsilon)*max(Q_next) + epsilon*mean(Q_next)`.
The update changes the selected action's weights toward that target. Step size is
normalized by the feature vector's squared length. Crashes, successful finishes, and deadline failures have no future term.
Two additional features give remaining route and time, so the model can
distinguish otherwise similar observations at different task stages.

Five driving actions × four sensing configurations = **20 joint choices**:

- `off`: all sensors in standby.
- `front`: forward RGB, event camera, and radar.
- `radar_rgb`: front radar and all eight RGB cameras.
- `all_on`: all 18 installed sensors.

This menu keeps the first experiment tractable. It does not yet learn individual
camera switches or every possible mask. The policy has no recurrent memory;
measurement aging supplies only limited temporal context.

In [ ]:
TRAIN_SEED = 42
TRAIN_EPISODES = 1000
DURATION_S = 120
ROUTE_DISTANCE_M = 2250
TEST_SEEDS = list(range(100_000, 100_004))
TEST_WEATHER = ["clear", "rain", "fog"]

config = ExperimentConfig()
config.highway["duration"] = DURATION_S
config.highway["high_speed_reward"] = 0.0  # Progress replaces the stock speed bonus.
config.highway["collision_reward"] = -10.0
config.weather.preset = "random"
config.weather.severity = 0.7
config.reward.route_distance_m = ROUTE_DISTANCE_M
config.reward.progress_weight = 0.02
config.reward.time_weight = 0.1
config.reward.completion_bonus = 10.0
config.reward.early_completion_bonus = 5.0
config.reward.failure_penalty = 10.0
settings = AgentSettings(episodes=TRAIN_EPISODES, alpha=0.12, gamma=0.99,
                         epsilon_start=1.0, epsilon_end=0.1)
# Fixed slow traffic isolates the new task; random background traffic is disabled.
config.highway["vehicles_count"] = 0
config.highway["slow_lead_scenario"] = [
    {"lane_offset": 0, "gap_m": 60.0, "speed_m_s": 12.0},
    {"lane_offset": 1, "gap_m": 180.0, "speed_m_s": 12.0},
    {"lane_offset": 2, "gap_m": 300.0, "speed_m_s": 12.0},
]
config.validate()
assert not set(TEST_SEEDS).intersection(range(TRAIN_SEED, TRAIN_SEED + TRAIN_EPISODES))
display(pd.Series(asdict(settings), name="Training settings"))
display(pd.Series(asdict(config.reward), name="Reward settings"))
print("Reward per decision = 0.02 * forward progress - 0.1 * elapsed seconds")
print("  - sensing energy/uncertainty costs + stock collision penalty")
print("  + 10 on safe finish + 5 * fraction of deadline remaining on safe finish")
print("  - 10 on crash/off-road/deadline failure")
print("2250 m at 30 m/s: 75 s; at 20 m/s: 112.5 s (idealized constant speeds).")
print("Finish detection occurs at decision boundaries (0.5 s).")


## Frozen-policy transfer check
Evaluate the completed original policies on the new traffic before training fresh agents.


In [ ]:
# Frozen old policies: test transfer before any new training.
BASE_RUN = PROJECT_ROOT / "runs/training/time_completion_120s_1000ep_20261121_204241_241658"
base_config = ExperimentConfig.load(BASE_RUN / "config.json")
base_adaptive = LinearExpectedSarsa.load(BASE_RUN / "adaptive_agent.npz")
base_all_on = LinearExpectedSarsa.load(BASE_RUN / "always_on_agent.npz")
base_agents = {"adaptive": base_adaptive, "learned_all_on": base_all_on}
transfer_evaluation = evaluate_agents(config, base_agents, TEST_SEEDS, TEST_WEATHER)
display(transfer_evaluation.groupby("policy").agg(
    completion_fraction=("completed", "mean"), crash_fraction=("crashed", "mean"),
    mean_leaders_passed=("slow_leaders_passed", "mean"), mean_power_w=("mean_power_w", "mean")))


## 3. Train the adaptive sensing agent

Exploration decreases linearly by episode. An epsilon of 1 means random choices;
0.1 means random choices 10% of the time. Every call starts fresh weights.

A zero Q estimate is not evidence of safety. Short training and sparse coverage
can leave poor actions overvalued. Training curves also mix learning with changing
exploration; use held-out evaluation to assess final behavior.

In [ ]:
adaptive_agent, adaptive_history = train_agent(config, settings, seed=TRAIN_SEED)
print("Weights:", adaptive_agent.weights.shape,
      "(joint actions × observation features)")
display(adaptive_history.tail())
CHECKPOINT_DIR = PROJECT_ROOT / "runs/local/overtaking_120s_2250m_1000ep_checkpoint"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
config.save(CHECKPOINT_DIR / "config.json")
adaptive_agent.save(CHECKPOINT_DIR / "adaptive_agent.npz")
adaptive_history.to_csv(CHECKPOINT_DIR / "adaptive_training.csv", index=False)


## 4. Train a learned always-on comparison

This agent uses the same features, episode budget, traffic seed range, learning
method, and reward. It chooses only driving actions; all sensors are always on.
It helps distinguish learning to drive from learning to manage sensing. Different
actions still produce different experience, so this is not an identical dataset.

In [ ]:
always_on_agent, always_on_history = train_agent(
    config, settings, seed=TRAIN_SEED, modes=("all_on",))
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for label, history in [("Adaptive sensing", adaptive_history), ("Learned always-on", always_on_history)]:
    for ax, metric in zip(axes, ["return", "crashed", "energy_j"]):
        rolling = history[metric].astype(float).rolling(min(10, TRAIN_EPISODES), min_periods=1).mean()
        ax.plot(history.episode, rolling, label=label)
        ax.set(xlabel="Episode", title=metric.replace("_", " "))
        ax.grid(alpha=.2)
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()
always_on_agent.save(CHECKPOINT_DIR / "always_on_agent.npz")
always_on_history.to_csv(CHECKPOINT_DIR / "always_on_training.csv", index=False)


## 5. Evaluate without exploration or learning

All four policies start from the same held-out traffic seeds for each condition:
learned adaptive sensing, learned always-on sensing, fixed IDLE/all-on, and random
driving/all-on. Evaluation uses epsilon=0, breaks value ties with a seeded RNG,
and does not change agent weights.

Report crashes, completion, distance, and sensing power together. **Lower total
energy can simply mean an earlier crash.** Mean power corrects for episode length
but does not establish safety. Total reward also depends on our chosen energy and
confidence penalties. The model can be confidently wrong.

Four seeds per weather condition are an exploratory check. They do not establish
statistical significance. Later use separate validation/test sets and multiple
independent training seeds.

**Interpretation:** lower crash rates can come from slower driving rather than
better sensing. A policy that spends most of its time with sensors off may have
learned a conservative speed pattern. Check sensing-mode use, speed, and actions;
these comparisons do not establish that sensor information caused an improvement.
Here completion means reaching 2,250 m safely, not surviving a time cutoff.
Compare **completion fraction and crash fraction first**, then completion time
among successful episodes. Failed episodes have missing completion times; a
policy that crashes early must not look fast. Elapsed time includes failures.
Raw return is not comparable with the old notebook because the task changed.
The new discount (0.99 versus 0.95) and task features also change the setup;
this run alone cannot isolate which change caused a behavior difference.


In [ ]:
agents = {"adaptive": adaptive_agent, "learned_all_on": always_on_agent,
          "idle_all_on": None, "random_all_on": None}
frozen_weights = adaptive_agent.weights.copy()
evaluation = evaluate_agents(config, agents, TEST_SEEDS, TEST_WEATHER)
assert np.array_equal(frozen_weights, adaptive_agent.weights)
summary = evaluation.groupby("policy", sort=False).agg(
    episodes=("seed", "count"), mean_leaders_passed=("slow_leaders_passed", "mean"), mean_return=("return", "mean"),
    return_std=("return", "std"), crash_fraction=("crashed", "mean"),
    completion_fraction=("completed", "mean"),
    mean_completion_time_s=("completion_time_s", "mean"),
    mean_elapsed_s=("elapsed_s", "mean"), mean_speed_m_s=("mean_speed_m_s", "mean"), mean_distance_m=("distance_m", "mean"),
    mean_power_w=("mean_power_w", "mean"), mean_energy_j=("energy_j", "mean"),
    mean_uncertainty=("mean_uncertainty", "mean"))
display(summary.round(3))
display(evaluation.groupby(["weather", "policy"], sort=False)[
    ["return", "crashed", "distance_m", "mean_power_w", "mean_uncertainty"]].mean().round(3))

In [ ]:
# Evaluate the newly trained policies on the unchanged original traffic too.
control_evaluation = evaluate_agents(base_config, agents, TEST_SEEDS, TEST_WEATHER)
display(control_evaluation.groupby("policy")[["completed", "crashed", "mean_power_w"]].mean())


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, metric, title in zip(axes,
        ["completion_fraction", "crash_fraction", "mean_completion_time_s", "mean_speed_m_s", "mean_power_w"],
        ["Completion fraction", "Crash fraction", "Successful finish time (s)", "Mean speed (m/s)", "Sensing power (W)"]):
    summary[metric].plot.bar(ax=ax, rot=35, title=title)
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=.2)
plt.tight_layout()
plt.show()
fractions = evaluation.groupby("policy", sort=False)[
    ["mode_fraction_off", "mode_fraction_front", "mode_fraction_radar_rgb", "mode_fraction_all_on"]].mean()
fractions.columns = [c.removeprefix("mode_fraction_") for c in fractions.columns]
fractions.plot.bar(stacked=True, figsize=(9, 3.5), rot=20,
                   title="Mean per-episode fraction of decisions in each sensing mode")
plt.ylabel("Fraction")
plt.tight_layout()
plt.show()

## 6. Inspect what the adaptive agent actually does

Watch one held-out scenario. The animation is illustrative; aggregate evaluation
above is the comparison. An initial blind action is expected: reset starts with no
measurements, and choosing sensing now affects only future observations.

Frequent switching can prevent sensors finishing warm-up or cancel pending
measurements. The environment charges active power while warming up. This short
agent has no explicit switching penalty or minimum dwell-time constraint.

In [ ]:
WATCH_SEED = TEST_SEEDS[0]
WATCH_WEATHER = "fog"
watch_config = copy.deepcopy(config)
watch_config.weather.preset = WATCH_WEATHER
watch_metrics, watch = run_agent(watch_config, adaptive_agent, seed=WATCH_SEED, capture=True)
display(pd.Series(watch_metrics, name="Watched episode"))
display(watch.history)
fig = plot_snapshot(watch)
plt.show()

In [ ]:
radius = watch.radius_m
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
road_image = axes[0].imshow(watch.frames[0])
axes[0].axis("off")
map_image = axes[1].imshow(np.ma.masked_where(watch.snapshots[0]["valid_cells"] == 0,
    watch.snapshots[0]["grid"][0]), extent=(-radius, radius, -radius, radius), vmin=0, vmax=1)
axes[1].plot(0, 0, "rx")
axes[1].set(xlabel="Right / E (m)", ylabel="Forward / N (m)")
fig.colorbar(map_image, ax=axes[1], shrink=.8)

def update_frame(i):
    road_image.set_data(watch.frames[i])
    obs = watch.snapshots[i]
    map_image.set_data(np.ma.masked_where(obs["valid_cells"] == 0, obs["grid"][0]))
    if i == 0:
        label = "Reset: sensors off, unknown map"
    else:
        row = watch.history.iloc[i - 1]
        label = f"t={row.time_s:.1f}s | {row.drive} | {row.sensing}"
    axes[0].set_title("Simulator visualization only")
    axes[1].set_title(label)
    return road_image, map_image

animation = FuncAnimation(fig, update_frame, frames=len(watch.frames),
                          interval=1000 / config.highway["policy_frequency"], blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))

## 7. Read the learned action values

These are estimated discounted returns, not safety probabilities. Below we rank
joint actions for the observation after the first decision (or the final available
observation). High values can be wrong with short training or an inadequate
feature representation. This inspection does not update the model.

In [ ]:
encoder = ObservationEncoder(config)
INSPECT_INDEX = min(1, len(watch.snapshots) - 1)
values = adaptive_agent.values(encoder(watch.snapshots[INSPECT_INDEX]))
action_values = pd.DataFrame([
    {"drive": drive, "sensing": mode, "estimated_Q": value}
    for (drive, mode), value in zip(adaptive_agent.actions, values)])
display(action_values.sort_values("estimated_Q", ascending=False).reset_index(drop=True).round(3))

## 8. Save this run

This saves both learned models, configuration, learning settings, training curves,
and held-out measurements together. Loading weights requires the same feature
schema, environment configuration, and action menu. A new timestamped directory
is created each time; saved runs are kept under `runs/`.

In [ ]:
SAVE_RUN = True
if SAVE_RUN:
    from datetime import datetime
    from zoneinfo import ZoneInfo
    stamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M%S_%f")
    output = PROJECT_ROOT / "runs/local" / f"overtaking_120s_2250m_1000ep_{stamp}"
    output.mkdir(parents=True, exist_ok=False)
    config.save(output / "config.json")
    adaptive_agent.save(output / "adaptive_agent.npz")
    always_on_agent.save(output / "always_on_agent.npz")
    adaptive_history.to_csv(output / "adaptive_training.csv", index=False)
    always_on_history.to_csv(output / "always_on_training.csv", index=False)
    evaluation.to_csv(output / "evaluation.csv", index=False)
    transfer_evaluation.to_csv(output / "frozen_transfer_evaluation.csv", index=False)
    control_evaluation.to_csv(output / "original_traffic_evaluation.csv", index=False)
    summary.to_csv(output / "summary.csv")
    watch.history.to_csv(output / "watched_episode.csv", index=False)
    (output / "metadata.json").write_text(json.dumps({
        "algorithm": "linear Expected SARSA", "feature_schema": adaptive_agent.feature_schema,
        "objective": "fixed-distance safe completion with early-finish bonus",
        "settings": asdict(settings), "training_seed": TRAIN_SEED,
        "test_seeds": TEST_SEEDS, "test_weather": TEST_WEATHER,
        "watch_seed": WATCH_SEED, "watch_weather": WATCH_WEATHER,
        "adaptive_actions": adaptive_agent.actions, "always_on_actions": always_on_agent.actions,
        "package_versions": PACKAGE_VERSIONS, "python": sys.version,
    }, indent=2))
    restored = LinearExpectedSarsa.load(output / "adaptive_agent.npz")
    assert np.array_equal(restored.weights, adaptive_agent.weights)
    print("Saved experiment:", output)
else:
    print("Saving disabled")

## Interpretation

Compare frozen-policy transfer with fresh training on the same held-out scenarios.
Report completion, crashes, leaders passed, completion time, and power together.
`slow_leaders_passed` counts leaders fully behind ego at episode end, across all
lanes; it is not a lane-change count or evidence of perception use. A failed trip
can still pass some leaders. Inspect replays and action fractions as well.

One thousand episodes use one training seed; four evaluation seeds per weather
remain exploratory. Repeat training seeds before claiming an improvement. There
is no extra passing reward: safe route completion remains the objective. After
this controlled test, vary gaps and speeds on separate held-out configurations,
then introduce braking vehicles and reactive background traffic as separate
changes. The fixed formation can itself be memorized, so success alone does not
show that sensing caused the maneuver.
